[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week08.ipynb)

# Week 8: Convolutional Networks I
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Convolution, pooling, and feature maps; building a CNN image classifier.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Build and train a CNN image classifier.
- Understand convolution, pooling, and feature maps.
- Compute how shapes and parameters change layer by layer.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. What a convolution does
A small learned filter slides over the input, sharing weights across positions.

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1)
x = torch.randn(1, 1, 8, 8)
print("input ", tuple(x.shape), "-> output", tuple(conv(x).shape))   # 4 feature maps, same H,W (padding=1)
print("filter weights:", tuple(conv.weight.shape), "(out, in, kh, kw)")

## 2. A hand-set edge filter
Convolution is intuitive: set a vertical-edge (Sobel) kernel and see it fire on edges.

In [ ]:
edge = torch.zeros(1, 1, 8, 8); edge[..., :, 4:] = 1.0      # left half 0, right half 1 (a vertical edge)
sobel = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
resp = F.conv2d(edge, sobel, padding=1)
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(edge[0, 0], cmap="gray"); ax[0].set_title("input (edge)")
ax[1].imshow(resp[0, 0]); ax[1].set_title("Sobel response (fires at the edge)")
for a in ax: a.axis("off")
plt.show()

## 3. Output size and parameter count
out = floor((in + 2p - k)/s) + 1 ; conv params = (k*k*Cin + 1)*Cout.

In [ ]:
c = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
print("output:", tuple(c(torch.randn(1, 3, 32, 32)).shape), " formula (32+2-3)/1+1 = 32")
print("params:", sum(p.numel() for p in c.parameters()), "= (3*3*3 + 1) * 16 =", (3 * 3 * 3 + 1) * 16)

## 4. Stride, padding, pooling change the size
A small table of input -> output shapes.

In [ ]:
x = torch.randn(1, 1, 32, 32)
configs = [("conv k3 s1 p1", nn.Conv2d(1, 1, 3, stride=1, padding=1)),
           ("conv k3 s2 p1", nn.Conv2d(1, 1, 3, stride=2, padding=1)),
           ("conv k5 s1 p0", nn.Conv2d(1, 1, 5, stride=1, padding=0)),
           ("maxpool 2",     nn.MaxPool2d(2)),
           ("avgpool 2",     nn.AvgPool2d(2))]
for name, layer in configs:
    print(f"{name:14s}: 32x32 -> {tuple(layer(x).shape)[2:]}")

## 5. CNN vs MLP parameter count on an image
Weight sharing is why CNNs are far cheaper on images.

In [ ]:
mlp_params = 28 * 28 * 128 + 128        # one dense layer 784 -> 128
cnn_params = (3 * 3 * 1 + 1) * 32        # one conv layer, 32 filters of 3x3
print(f"dense 784->128 : {mlp_params:,} params")
print(f"conv 1->32 (3x3): {cnn_params:,} params  (reused at every position)")

## 6. Build a CNN and trace shapes
Stack conv/pool blocks; channels grow while spatial size shrinks.

In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 28 -> 14
    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 14 -> 7
    nn.Flatten(), nn.Linear(16 * 7 * 7, 10))
x = torch.randn(4, 1, 28, 28)
for layer in cnn:
    x = layer(x); print(f"{layer.__class__.__name__:9s}", tuple(x.shape))

## 7. Train on FashionMNIST
A couple of hundred steps is enough to see it learn (downloads ~30 MB).

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
train = datasets.FashionMNIST("./data", train=True, download=True, transform=transforms.ToTensor())
test  = datasets.FashionMNIST("./data", train=False, download=True, transform=transforms.ToTensor())
dl = DataLoader(train, batch_size=128, shuffle=True)
model = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                      nn.Flatten(), nn.Linear(8 * 14 * 14, 10)).to(device)
opt = torch.optim.Adam(model.parameters(), 1e-3); f = nn.CrossEntropyLoss()
for step, (xb, yb) in enumerate(dl):
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad(); loss = f(model(xb), yb); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f"step {step}: loss {loss.item():.3f}")
    if step == 200:
        break

## 8. Measure test accuracy
Evaluate on the held-out test split in eval mode.

In [ ]:
test_dl = DataLoader(test, batch_size=512)
model.eval(); correct = total = 0
with torch.no_grad():
    for xb, yb in test_dl:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item(); total += yb.numel()
print(f"test accuracy after 200 steps: {correct/total:.3f}")

## 9. Visualize the first-layer feature maps
Each map is one filter's response across the image.

In [ ]:
fmap = model[0](xb[:1]).detach().cpu()
fig, ax = plt.subplots(1, 8, figsize=(12, 2))
for i in range(8):
    ax[i].imshow(fmap[0, i]); ax[i].axis("off")
plt.suptitle("First-conv feature maps"); plt.show()

**Mini-exercise:** add a second conv block (8 -> 16 channels with a pool) before the linear head and retrain. Does test accuracy improve for the same number of steps? **Recap:** convolution is local connectivity with shared weights; out = floor((in + 2p - k)/s) + 1; deeper feature maps are more abstract.

---
Student materials for this week: the lab handout (`labs/week08.html`) and the curated references (`references/week08.html`) in the course site.